In [1]:
from ultralytics import YOLO
from pathlib import Path

# Absolute paths (Windows)
model_path = Path("best.pt")
# model_path1 = Path("yolo26n.pt")
# source_dir = Path(r"C:\Users\YASH JAIN\Documents\New Folder\OneDrive\Desktop\pothole\Pothole_Detection.v1i.yolo26\test\images")

# Output folder
# project_dir = Path(r"C:\Users\YASH JAIN\Documents\New Folder\OneDrive\Desktop\pothole\runs\detect")
# run_name = "pothole_test_best"

# Load and predict
model = YOLO(str(model_path))
# results = model.predict(
#     # source=str(source_dir),
#     imgsz=640,
#     conf=0.25,
#     save=True,
#     # project=str(project_dir),
#     name=run_name,
#     exist_ok=True
# )


print("Prediction complete.")
# print(f"Saved in: {project_dir / run_name}")

Prediction complete.


In [4]:
# Test set report + accuracy (YOLO metrics)
from pathlib import Path
import yaml

# Dataset YAML path
data_yaml = Path(r"C:\Users\YASH JAIN\Documents\New Folder\OneDrive\Desktop\pothole\hole\data.yaml")

if not data_yaml.exists():
    raise FileNotFoundError(f"Dataset config not found: {data_yaml.resolve()}")

# Load dataset config
with open(data_yaml, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}

nc = int(cfg.get("nc", 0))
if nc <= 0:
    raise ValueError("Invalid 'nc' in data.yaml")

# Resolve split paths robustly (handles wrong ../ in yaml)
base_dir = data_yaml.parent
root_in_yaml = cfg.get("path", "")
if root_in_yaml:
    root_dir = Path(root_in_yaml)
    if not root_dir.is_absolute():
        root_dir = (base_dir / root_dir).resolve()
else:
    root_dir = base_dir

def resolve_split_path(raw_split: str) -> Path:
    p = Path(str(raw_split))
    candidates = []

    if p.is_absolute():
        candidates.append(p)
    else:
        # Standard YOLO behavior: relative to root_dir
        candidates.append((root_dir / p).resolve())
        # Fallback: relative to data.yaml folder
        candidates.append((base_dir / p).resolve())
        # Fallback: strip leading ../ or ./ if dataset was moved
        cleaned = [part for part in p.parts if part not in ("..", ".")]
        if cleaned:
            candidates.append((base_dir / Path(*cleaned)).resolve())

    for c in candidates:
        if c.exists():
            return c

    # return best guess (first candidate) for clear error message
    return candidates[0]

train_path = resolve_split_path(cfg.get("train", "")) if cfg.get("train") else None
val_path = resolve_split_path(cfg.get("val", "")) if cfg.get("val") else None
test_path = resolve_split_path(cfg.get("test", "")) if cfg.get("test") else None

if test_path is None or not test_path.exists():
    raise FileNotFoundError(f"Resolved test images folder not found: {test_path}")

# Convert test images path -> labels path
def image_to_labels_dir(images_dir: Path) -> Path:
    s = str(images_dir)
    if "images" in s:
        return Path(s.replace("images", "labels"))
    return images_dir.parent / "labels"

labels_dir = image_to_labels_dir(test_path)
if not labels_dir.exists():
    raise FileNotFoundError(f"Test labels folder not found: {labels_dir}")

# Validate class IDs in test labels to prevent IndexError in val()
invalid = []
for lf in labels_dir.rglob("*.txt"):
    with open(lf, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            p = line.strip().split()
            if not p:
                continue
            try:
                cls_id = int(float(p[0]))
            except ValueError:
                invalid.append((str(lf), ln, p[0], "not-numeric"))
                continue
            if cls_id < 0 or cls_id >= nc:
                invalid.append((str(lf), ln, cls_id, f"valid range: 0..{nc-1}"))

if invalid:
    print("Found invalid class IDs in test labels (showing first 10):")
    for item in invalid[:10]:
        print(item)
    raise ValueError(
        f"Label class IDs are out of range for nc={nc}. "
        "Fix labels or update nc/names in data.yaml."
    )

# Build a temporary resolved YAML for stable validation
data_resolved = {
    "path": str(root_dir),
    "train": str(train_path) if train_path else cfg.get("train"),
    "val": str(val_path) if val_path else cfg.get("val"),
    "test": str(test_path),
    "nc": nc,
    "names": cfg.get("names", []),
}

resolved_yaml = data_yaml.with_name("data_resolved_for_val.yaml")
with open(resolved_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_resolved, f, sort_keys=False)

print(f"Using test images: {test_path}")
print(f"Using test labels: {labels_dir}")

# Evaluate on test split
metrics = model.val(
    data=str(resolved_yaml),
    split="test",
    imgsz=640,
    conf=0.25,
    save_json=True,
    plots=False,
    verbose=False,
)

# Extract key detection metrics
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map50_95 = metrics.box.map

print("\n" + "=" * 60)
print("TESTING REPORT")
print("=" * 60)
print(f"Model: {model_path}")
print(f"Dataset YAML used: {resolved_yaml.resolve()}")
print("-" * 60)
print(f"Precision:        {precision:.4f}")
print(f"Recall:           {recall:.4f}")
print(f"mAP@0.50:         {map50:.4f}")
print(f"mAP@0.50:0.95:    {map50_95:.4f}")
print("=" * 60)
print(f"Testing Accuracy (mAP@0.50): {map50 * 100:.2f}%")
print("=" * 60)

Using test images: C:\Users\YASH JAIN\Documents\New Folder\OneDrive\Desktop\pothole\hole\test\images
Using test labels: C:\Users\YASH JAIN\Documents\New Folder\OneDrive\Desktop\pothole\hole\test\labels
Ultralytics 8.4.34  Python-3.13.11 torch-2.11.0+cpu CPU (11th Gen Intel Core i5-11320H @ 3.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 81.622.0 MB/s, size: 49.8 KB)
val: Scanning C:\Users\YASH JAIN\Documents\New Folder\OneDrive\Desktop\pothole\hole\test\labels.cache... 198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 198/198 455.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.5s/it 31.9s2.4s
                   all        198        433      0.889      0.847      0.918      0.682
Speed: 3.8ms preprocess, 147.3ms inference, 0.0ms loss, 0.9ms postprocess per image
Saving C:\Users\YASH JAIN\Documents\New Folder\OneDr

## Pothole detection form image


In [ ]:
# Pothole Detection and Report Generation - User Uploaded Image
from pathlib import Path
from PIL import Image
import cv2
import numpy as np
from ipywidgets import FileUpload, Output
from IPython.display import display
import matplotlib.pyplot as plt
from datetime import datetime

# Create file upload widget
uploader = FileUpload(accept='image/*', multiple=False, description='Upload Image')
output = Output()

def on_upload_change(change):
    output.clear_output()
    with output:
        if uploader.value:
            uploaded_file = uploader.value[0]
            file_name = uploaded_file['name']
            file_content = uploaded_file['content']
            
            
            print(f"Processing image: {file_name}\n")
            
            temp_path = Path("temp_upload.jpg")

            # Save file
            with open(temp_path, 'wb') as f:
                f.write(bytes(file_content))
            
            try:
                # 🔹 Prediction
                results = model.predict(source=str(temp_path), conf=0.2, imgsz=640)
                result = results[0]

                pothole_count = len(result.boxes) if result.boxes is not None else 0

                # 📝 Build report
                report_lines = []
                report_lines.append("=" * 60)
                report_lines.append("POTHOLE DETECTION REPORT")
                report_lines.append("=" * 60)
                report_lines.append(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                report_lines.append(f"Image Name: {file_name}")
                report_lines.append(f"Total Potholes Detected: {pothole_count}")
                report_lines.append("=" * 60)

                print("\n".join(report_lines))

                # 🔹 Details
                if pothole_count > 0:
                    report_lines.append("\nDetection Details:")
                    report_lines.append("-" * 60)
                    print("\nDetection Details:")
                    print("-" * 60)

                    for i, box in enumerate(result.boxes, 1):
                        conf = box.conf.item()
                        x1, y1, x2, y2 = box.xyxy[0]
                        width = x2 - x1
                        height = y2 - y1
                        detail = f"Pothole {i}: Confidence = {conf:.2%}, Width = {width:.0f}px, Height = {height:.0f}px"
                        report_lines.append(detail)
                        print(detail)

                else:
                    report_lines.append("\nNo potholes detected")
                    print("No potholes detected")

                # 📁 Save report to text file
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                report_file = Path(f"IN_image/pothole_report_{timestamp}.txt")
                with open(report_file, 'w', encoding='utf-8') as rf:
                    rf.write("\n".join(report_lines))
                
                report_lines.append("\n" + "=" * 60)
                report_lines.append(f"Report saved to: {report_file.resolve()}")
                report_lines.append("=" * 60)
                print(f"\nReport saved to: {report_file.resolve()}")

                # ===============================
                # 🔥 UPDATED IMAGE DISPLAY PART
                # ===============================
                print("\nDetected Image:")

                # 🔹 Original image load
                img = cv2.imread(str(temp_path))

                # 🔹 Loop through detections
                if result.boxes is not None:
                    for box in result.boxes:
                        x1, y1, x2, y2 = map(int, box.xyxy[0])
                        conf = box.conf.item()

                        # Draw RED bounding box
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 1)

                        # Label text
                        label = f"pothole {conf:.2f}"

                        # 🔹 Put text (small size, no background)
                        cv2.putText(
                            img,
                            label,
                            (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.3,              
                            (0, 0,255),      
                            1,
                            cv2.LINE_AA
                        )

                # 🔹 Convert BGR → RGB
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # 🔹 Show image
                plt.figure(figsize=(8, 6))
                plt.imshow(img)
                plt.axis("off")
                plt.show()

            except Exception as e:
                print(f"Error: {e}")

            finally:
                if temp_path.exists():
                    temp_path.unlink()

# Attach callback
uploader.observe(on_upload_change, names='value')

# Display
print("Upload an image to detect potholes:")
display(uploader)
display(output)

Upload an image to detect potholes:


FileUpload(value=(), accept='image/*', description='Upload Image')

Output()

## For live web cam


In [5]:
import cv2
import time
import numpy as np
from pathlib import Path
from datetime import datetime

# model already loaded

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Cannot access webcam")
    exit()

frame_width = int(cap.get(3))
frame_height = int(cap.get(4))

out = cv2.VideoWriter(
    "output.avi",
    cv2.VideoWriter_fourcc(*'XVID'),
    20,
    (frame_width, frame_height)
)

#Report variables

total_detected_frames = 0
frame_number = 0

# Unique pothole tracking
unique_pothole_ids = set()
next_pothole_id = 1
previous_centroids = {}
distance_threshold = 50  # pixels

print("Press 'q' to stop and see report...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_number += 1
    current_centroids = {}

    try:
        results = model.predict(source=frame, conf=0.25, imgsz=640)
        result = results[0]

        count = len(result.boxes) if result.boxes is not None else 0

        # 🔥 Update report data
        if count > 0:
            total_detected_frames += 1
        

        # 🔴 Draw boxes and track unique potholes
        if result.boxes is not None:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = box.conf.item()

                # Calculate centroid
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                centroid = (cx, cy)

                # 🆔 Match to previous detections
                matched = False
                for prev_id, prev_centroid in previous_centroids.items():
                    distance = np.sqrt((cx - prev_centroid[0])**2 + (cy - prev_centroid[1])**2)
                    if distance < distance_threshold:
                        # Same pothole detected again
                        pothole_id = prev_id
                        matched = True
                        break

                if not matched:
                    # unique pothole
                    pothole_id = next_pothole_id
                    next_pothole_id += 1
                    unique_pothole_ids.add(pothole_id)

                current_centroids[pothole_id] = centroid

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

                label = f"ID:{pothole_id} {conf:.2f}"
                cv2.putText(frame, label, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (0, 0, 255), 1, cv2.LINE_AA)

        # Update previous centroids for next frame
        previous_centroids = current_centroids

        # 🔢 Live count
        cv2.putText(frame, f"Current: {count} | Unique: {len(unique_pothole_ids)}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # 💾 Save image
        # if count > 0:
        #     cv2.imwrite(f"detected_{int(time.time())}.jpg", frame)

        # 🎥 Save video
        out.write(frame)

        cv2.imshow("Live Pothole Detection", frame)

    except Exception as e:
        print(f"Error: {e}")

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 🔹 Release
cap.release()
out.release()
cv2.destroyAllWindows()

# =========================
# 📊 FINAL REPORT
# =========================
report_lines = [
    "="*50,
    "FINAL POTHOLE DETECTION REPORT",
    "="*50,
    f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Total Frames Processed: {frame_number}",
    f"Unique Potholes in Session: {len(unique_pothole_ids)}",
    f"Frames with Potholes: {total_detected_frames}",
    "Potholes were detected in this session." if total_detected_frames > 0 else "No potholes detected.",
    "="*50
]

print("\n" + "\n".join(report_lines))

report_file = Path(f"IN_cam/live_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
with open(report_file, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"Report saved to: {report_file.resolve()}")

Press 'q' to stop and see report...

0: 480x640 (no detections), 152.2ms
Speed: 8.0ms preprocess, 152.2ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 119.5ms
Speed: 3.9ms preprocess, 119.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 116.0ms
Speed: 2.9ms preprocess, 116.0ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 pothole, 95.9ms
Speed: 6.4ms preprocess, 95.9ms inference, 2.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 88.3ms
Speed: 3.1ms preprocess, 88.3ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 98.0ms
Speed: 3.4ms preprocess, 98.0ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 89.0ms
Speed: 3.7ms preprocess, 89.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 pothole, 88.0ms
Spe

## for viedo


In [ ]:
import cv2
import time
import numpy as np
from pathlib import Path
from datetime import datetime

# 🔹 Load model
# model = YOLO("best.pt")

# 🔹 Input video path
video_path = r"C:\Users\YASH JAIN\Downloads\videoplayback (1).mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Cannot open video")
    exit()

# 🔹 Video properties
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# 🔹 Output video
out = cv2.VideoWriter(
    "output_video.avi",
    cv2.VideoWriter_fourcc(*'XVID'),
    fps,
    (frame_width, frame_height)
)

# 🔥 Report variables

total_detected_frames = 0

frame_number = 0

# 🆔 Unique pothole tracking
unique_pothole_ids = set()
next_pothole_id = 1
previous_centroids = {}
distance_threshold = 50  # pixels

print("Processing video... Press 'q' to stop")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_number += 1
    current_centroids = {}
    model
    try:
        # 🔹 Prediction
        results = model.predict(source=frame, conf=0.25, imgsz=640)
        result = results[0]

        count = len(result.boxes) if result.boxes is not None else 0

        # 🔥 Update report
        if count > 0:
            total_detected_frames += 1
        

        # 🔴 Draw detections and track unique potholes
        if result.boxes is not None:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = box.conf.item()

                # Calculate centroid
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                centroid = (cx, cy)

                # 🆔 Match to previous detections
                matched = False
                for prev_id, prev_centroid in previous_centroids.items():
                    distance = np.sqrt((cx - prev_centroid[0])**2 + (cy - prev_centroid[1])**2)
                    if distance < distance_threshold:
                        # Same pothole detected again
                        pothole_id = prev_id
                        matched = True
                        break

                if not matched:
                    # New unique pothole
                    pothole_id = next_pothole_id
                    next_pothole_id += 1
                    unique_pothole_ids.add(pothole_id)

                current_centroids[pothole_id] = centroid

                # Draw bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

                # Draw pothole ID
                label = f"ID:{pothole_id} {conf:.2f}"
                cv2.putText(frame, label, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (0, 0, 255), 1, cv2.LINE_AA)

        # Update previous centroids for next frame
        previous_centroids = current_centroids

        # 🔢 Show count
        cv2.putText(frame, f"Current: {count} | Unique: {len(unique_pothole_ids)}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # 💾 Save frame if pothole detected
        # if count > 0:
        #     cv2.imwrite(f"frame_{frame_number}.jpg", frame)

        # 🎥 Write video
        out.write(frame)

        # 🔹 Show video
        cv2.imshow("Video Pothole Detection", frame)

    except Exception as e:
        print(f"Error: {e}")

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 🔹 Release
cap.release()
out.release()
cv2.destroyAllWindows()

# =========================
# 📊 FINAL REPORT
# =========================
report_lines = [
    "="*50,
    "VIDEO POTHOLE DETECTION REPORT",
    "="*50,
    f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Total Frames Processed: {frame_number}",
    f"Unique Potholes in Video: {len(unique_pothole_ids)}",
    f"Frames with Potholes: {total_detected_frames}",
    "Potholes detected in video." if total_detected_frames > 0 else "No potholes detected.",
    "="*50
]

print("\n" + "\n".join(report_lines))

report_file = Path(f"IN_vedio/video_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
with open(report_file, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"Report saved to: {report_file.resolve()}")

Error: Cannot open video
Processing video... Press 'q' to stop

VIDEO POTHOLE DETECTION REPORT
Timestamp: 2026-04-13 08:23:08
Total Frames Processed: 0
Unique Potholes in Video: 0
Frames with Potholes: 0
No potholes detected.


FileNotFoundError: [Errno 2] No such file or directory: 'IN_vedio\\video_report_20260413_082308.txt'

: 